# 03. Graph + Vector — GraphRAG

Apache AGE 그래프 + pgvector 벡터 검색을 하나의 PostgreSQL에서 결합.

**사전 준비**
```bash
cd docker && docker compose up -d
pip install "langchain-age[all]" langchain-openai
```

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 그래프에 지식 구축

In [ ]:
from langchain_age import AGEGraph

graph = AGEGraph(DSN, graph_name="techkg")

nodes = [
    ("PostgreSQL", "Database", "Open-source relational database with extensibility"),
    ("Apache AGE", "Extension", "Graph query extension for PostgreSQL using Cypher"),
    ("pgvector", "Extension", "Vector similarity search extension for PostgreSQL"),
    ("LangChain", "Framework", "Framework for building applications with LLMs"),
    ("Neo4j", "Database", "Native graph database with Bolt protocol"),
    ("Python", "Language", "General-purpose programming language for AI and data science"),
]
for name, label, description in nodes:
    graph.query(f"MERGE (n:{label} {{name: '{name}', description: '{description}'}})")

rels = [
    ("Apache AGE", "Extension", "PostgreSQL", "Database", "EXTENDS"),
    ("pgvector", "Extension", "PostgreSQL", "Database", "EXTENDS"),
    ("LangChain", "Framework", "Python", "Language", "BUILT_WITH"),
    ("LangChain", "Framework", "Neo4j", "Database", "INTEGRATES_WITH"),
    ("LangChain", "Framework", "PostgreSQL", "Database", "INTEGRATES_WITH"),
]
for src, sl, tgt, tl, rt in rels:
    graph.query(f"MATCH (a:{sl} {{name: '{src}'}}), (b:{tl} {{name: '{tgt}'}}) MERGE (a)-[:{rt}]->(b)")

graph.refresh_schema()
print(graph.schema)

## 2. 그래프 노드 → 벡터화

In [ ]:
from langchain_age import AGEVector

db_store = AGEVector.from_existing_graph(
    embedding=embeddings,
    connection_string=DSN,
    graph_name="techkg",
    node_label="Database",
    text_node_properties=["name", "description"],
    collection_name="techkg_databases",
)

ext_store = AGEVector.from_existing_graph(
    embedding=embeddings,
    connection_string=DSN,
    graph_name="techkg",
    node_label="Extension",
    text_node_properties=["name", "description"],
    collection_name="techkg_extensions",
)

print(f"Databases: {len(db_store.similarity_search('x', k=100))} vectors")
print(f"Extensions: {len(ext_store.similarity_search('x', k=100))} vectors")

## 3. 벡터 검색 → 그래프 컨텍스트 확장

In [ ]:
results = db_store.similarity_search_with_relevance_scores("graph query engine", k=2)

for doc, score in results:
    print(f"\n[{score:.3f}] {doc.page_content}")

    # 그래프에서 관계 확장
    label = doc.metadata.get("node_label", "Database")
    name = doc.page_content.split()[0]
    neighbors = graph.query(
        f"MATCH (n:{label} {{name: '{name}'}})<-[r]-(m) "
        f"RETURN type(r) AS rel, labels(m) AS labels, m.name AS neighbor"
    )
    for n in neighbors:
        print(f"  <-[{n['rel']}]- {n['neighbor']}")

## 4. Cypher QA Chain

LLM이 자연어 → Cypher → AGE 실행 → 자연어 답변 파이프라인.

In [ ]:
from langchain_age import AGEGraphCypherQAChain

chain = AGEGraphCypherQAChain.from_llm(
    llm,
    graph=graph,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
    verbose=True,
)

questions = [
    "What extensions does PostgreSQL have?",
    "What is LangChain built with?",
    "Which databases does LangChain integrate with?",
]

for q in questions:
    result = chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print(f"   Cypher: {result['intermediate_steps'][0]['query']}")

## 5. 정리

In [ ]:
db_store._drop_table()
ext_store._drop_table()
db_store.close()
ext_store.close()
graph.close()
print("Done.")